# 16 — 2023 Ward Name Matching

This notebook resolves the 2023 House of Commons Library local-election rows that do not carry ONS ward codes.

The objective is to match:

`source council name + source ward name`

onto:

`LAD23CD + LAD23NM + WD23CD + WD23NM`

using the `oa21_to_wd23_lad23_eng_wal.csv` lookup file.

The notebook is deliberately national. It will attempt automatic matching for the full 2023 dataset, then produce review files for uncertain cases.

## Matching strategy

1. Build a unique 2023 ward candidate list from the OA21→WD23 lookup.
2. Clean council and ward names into standard comparison strings.
3. Run exact matching first.
4. Run fuzzy matching second.
5. Auto-approve only very high-confidence matches.
6. Send ambiguous/low-confidence rows to manual review.
7. Optionally patch `ward_result_summary_v1.csv` into a new file with approved 2023 matches applied.

This notebook does **not** overwrite the original ward-result summary. It writes a new `ward_result_summary_v2_with_2023_matches.csv` file.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib

try:
    from rapidfuzz import fuzz, process
    RAPIDFUZZ_AVAILABLE = True
except Exception:
    import difflib
    RAPIDFUZZ_AVAILABLE = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

ELECTION_DIR = PROJECT_DIR / "data" / "processed" / "election_results"
GEOGRAPHY_DIR = PROJECT_DIR / "data" / "geography"
DICTIONARY_DIR = PROJECT_DIR / "data" / "dictionaries"
OUTPUT_DIR = ELECTION_DIR / "name_matching_2023_v1"

DICTIONARY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NAME_MATCH_REQUIRED_PATH = ELECTION_DIR / "election_name_match_required_v1.csv"
WARD_SUMMARY_PATH = ELECTION_DIR / "ward_result_summary_v1.csv"
LOOKUP_2023_PATH = GEOGRAPHY_DIR / "oa21_to_wd23_lad23_eng_wal.csv"
WARD_NAME_DICTIONARY_PATH = DICTIONARY_DIR / "ward_name_dictionary_v1.csv"

print("Project directory:", PROJECT_DIR)
print("Election directory:", ELECTION_DIR)
print("Geography directory:", GEOGRAPHY_DIR)
print("Dictionary directory:", DICTIONARY_DIR)
print("Output directory:", OUTPUT_DIR)

for p in [NAME_MATCH_REQUIRED_PATH, WARD_SUMMARY_PATH, LOOKUP_2023_PATH]:
    print(p.name, "exists:", p.exists())
    if not p.exists():
        raise FileNotFoundError(p)

print("RapidFuzz available:", RAPIDFUZZ_AVAILABLE)

Project directory: c:\Users\keena\Documents\Electoral_Tribes
Election directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results
Geography directory: c:\Users\keena\Documents\Electoral_Tribes\data\geography
Dictionary directory: c:\Users\keena\Documents\Electoral_Tribes\data\dictionaries
Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\name_matching_2023_v1
election_name_match_required_v1.csv exists: True
ward_result_summary_v1.csv exists: True
oa21_to_wd23_lad23_eng_wal.csv exists: True
RapidFuzz available: True


## 1. Load 2023 name-only result areas

The source file should be `election_name_match_required_v1.csv`, generated by Notebook 13.

Only `source_year == 2023` rows are handled here.

In [4]:
name_required = pd.read_csv(NAME_MATCH_REQUIRED_PATH, low_memory=False)

name_2023 = name_required[name_required["source_year"].astype(str).eq("2023")].copy()

print("All name-match-required rows:", len(name_required))
print("2023 rows:", len(name_2023))
print("Unique result areas:", name_2023["result_area_key"].nunique())

required_cols = ["result_area_key", "source_year", "council_name", "ward_name"]
missing = [c for c in required_cols if c not in name_2023.columns]
if missing:
    raise ValueError(f"Missing required columns from name_required: {missing}")

display(name_2023.head())

All name-match-required rows: 4831
2023 rows: 4831
Unique result areas: 4831


,result_area_key,source_year,election_year,election_date,source_file,council_name,lad_code,ward_name,standard_ward_name,ward_code,ec_ward_code,boundary_year,geography_type,atlas_join_strategy,election_type,ordinary_or_by_election,seats_available,electorate,turnout,valid_votes,ballots,invalid_votes,candidate_count,top_party_by_votes,runner_up_party_by_votes,top_party_votes,runner_up_party_votes,margin_votes,margin_pct,top_party_vote_share,runner_up_vote_share,elected_parties,elected_candidates,lowest_elected_votes,first_losing_candidate,first_losing_party,first_losing_votes,con_votes,lab_votes,ld_votes,green_votes,reform_ukip_brexit_votes,independent_votes,sdp_votes,other_votes,con_share,lab_share,ld_share,green_share,reform_ukip_brexit_share,independent_share,sdp_share,other_share,party_fragmentation_index,effective_number_of_parties,data_quality_flag,manual_review_required,candidate_effective_total_votes,source_geography_code,source_geography_name,source_boundary_year,detected_geography_type,atlas_join_ready,join_readiness_status,recommended_next_action,source_code_found_in_lookup
0,2023|NAME|AMBER_VALLEY|ALFRETON,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,ALFRETON,ALFRETON,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,3.0,6673.0,24.7,1675.0,NaN,NaN,13,Labour,Conservative,828,443.0,385.0,0.229851,0.494328,0.264478,Labour; Labour; Labour,"Marshall-Clarke, S.; Dolman, G.; Wood, K.",763,"Boyce, C.",Conservative,443.0,443,828,104,157,143,0,0,0,0.264478,0.494328,0.062090,0.093731,0.085373,0.0,0.0,0.000000,0.665762,2.991878,needs_ward_code_match,True,1675,NaN,ALFRETON,2023,name_only_no_ons_code,False,name_match_required,review_ward_name_dictionary,False
1,2023|NAME|AMBER_VALLEY|ALPORT_AND_SOUTH_WEST_P...,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,Alport & South West Parishes,Alport & South West Parishes,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,2.0,4569.0,41.9,2084.0,NaN,NaN,8,Conservative,Green,868,496.0,372.0,0.178503,0.416507,0.238004,Conservative; Conservative,"Taylor, D.; Orton, J.",846,"Meynell, G.",Green,496.0,868,455,165,496,100,0,0,0,0.416507,0.218330,0.079175,0.238004,0.047985,0.0,0.0,0.000000,0.713637,3.492073,needs_ward_code_match,True,2084,NaN,Alport & South West Parishes,2023,name_only_no_ons_code,False,name_match_required,review_ward_name_dictionary,False
2,2023|NAME|AMBER_VALLEY|BELPER_EAST,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,Belper East,Belper East,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,3.0,8046.0,40.1,3438.0,NaN,NaN,15,Labour,BELPER IND,978,890.0,88.0,0.025596,0.284468,0.258871,Labour; Labour; BELPER IND,"Hill, T.; Porter, J.; Atkinson, F.",890,"Nelson, J.",Conservative,867.0,867,978,234,469,0,0,0,890,0.252182,0.284468,0.068063,0.136417,0.000000,0.0,0.0,0.258871,0.765226,4.259419,needs_ward_code_match,True,3438,NaN,Belper East,2023,name_only_no_ons_code,False,name_match_required,review_ward_name_dictionary,False
3,2023|NAME|AMBER_VALLEY|BELPER_NORTH,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,Belper North,Belper North,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,2.0,4786.0,45.0,2356.0,NaN,NaN,8,Labour,BELPER IND,783,663.0,120.0,0.050934,0.332343,0.281409,Labour; BELPER IND,"Monkman, E.; Bellamy, B.",663,"Butler, J.",Labour,600.0,570,783,0,340,0,0,0,663,0.241935,0.332343,0.000000,0.144312,0.000000,0.0,0.0,0.281409,0.730998,3.717447,needs_ward_code_match,True,2356,NaN,Belper North,2023,name_only_no_ons_code,False,name_match_required,review_ward_name_dictionary,False
4,2023|NAME|AMBER_VALLEY|BELPER_SOUTH,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,BELPER SOUTH,BELPER SOUTH,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,2.0,4342.0,40.0,1468.0,NaN,NaN,9,Green,Conservative,826,268.0,558.0,0.380109,0.562670,0.182561,Green; Green,"Kinsella, G.; Walls, J.",790,"Miller, B.",Con

## 2. Load and standardise the 2023 OA→Ward→LAD lookup

The lookup is OA-level, so this notebook collapses it to a unique list of 2023 wards.

Expected lookup columns are some variant of:

- `OA21CD`
- `WD23CD`
- `WD23NM`
- `LAD23CD`
- `LAD23NM`

In [5]:
def normalise_col_name(col):
    return re.sub(r"[^a-z0-9]+", "", str(col).lower())


def find_column(df, candidates, required=True):
    normalised = {normalise_col_name(c): c for c in df.columns}
    for cand in candidates:
        key = normalise_col_name(cand)
        if key in normalised:
            return normalised[key]
    if required:
        raise KeyError(f"None of these columns found: {candidates}. Available columns: {df.columns.tolist()}")
    return None


def standardise_oa_to_wd23_lookup(path):
    lookup = pd.read_csv(path, low_memory=False)

    col_map = {
        find_column(lookup, ["OA21CD", "OA21_CODE", "OA21 code"]): "OA21CD",
        find_column(lookup, ["WD23CD", "WD23_CODE", "Ward code", "Ward Code"]): "WD23CD",
        find_column(lookup, ["WD23NM", "WD23_NAME", "Ward name", "Ward Name"]): "WD23NM",
        find_column(lookup, ["LAD23CD", "LAD23_CODE", "Local authority code", "LAD code"]): "LAD23CD",
        find_column(lookup, ["LAD23NM", "LAD23_NAME", "Local authority name", "LAD name"]): "LAD23NM",
    }

    lookup = lookup.rename(columns=col_map)

    keep = ["OA21CD", "WD23CD", "WD23NM", "LAD23CD", "LAD23NM"]
    lookup = lookup[keep].copy()

    for col in keep:
        lookup[col] = lookup[col].astype(str).str.strip()
        lookup.loc[lookup[col].isin(["nan", "None", ""]), col] = np.nan

    return lookup

lookup_2023 = standardise_oa_to_wd23_lookup(LOOKUP_2023_PATH)

ward_candidates = (
    lookup_2023[["LAD23CD", "LAD23NM", "WD23CD", "WD23NM"]]
    .dropna(subset=["LAD23CD", "LAD23NM", "WD23CD", "WD23NM"])
    .drop_duplicates()
    .copy()
)

print("OA lookup rows:", len(lookup_2023))
print("Unique 2023 ward candidates:", len(ward_candidates))
print("Unique LADs:", ward_candidates["LAD23CD"].nunique())

display(ward_candidates.head())

OA lookup rows: 188880
Unique 2023 ward candidates: 7608
Unique LADs: 318


,LAD23CD,LAD23NM,WD23CD,WD23NM
0,E06000001,Hartlepool,E05013038,Burn Valley
26,E06000001,Hartlepool,E05013039,De Bruce
50,E06000001,Hartlepool,E05013049,Victoria
73,E06000002,Middlesbrough,E05009853,Acklam
90,E06000002,Middlesbrough,E05009854,Ayresome


## 3. Cleaning functions for council and ward names

These functions deliberately remove punctuation, case differences, common local-authority suffixes, and `&`/`and` differences.

They do **not** remove substantive words from ward names, because doing that would create unsafe false matches.

In [6]:
def clean_text_base(value):
    if pd.isna(value):
        return ""

    text = str(value).lower().strip()
    text = text.replace("&", " and ")
    text = text.replace("+", " and ")
    text = re.sub(r"st[.]?", "saint", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_council_name(value):
    text = clean_text_base(value)

    # Remove administrative suffixes often present in one source but not another.
    suffix_phrases = [
        "city council", "borough council", "district council", "county council",
        "metropolitan borough council", "unitary authority", "council",
        "city of", "district of", "borough of", "royal borough of",
    ]

    for phrase in suffix_phrases:
        text = re.sub(rf"{re.escape(phrase)}", " ", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_ward_name(value):
    text = clean_text_base(value)

    # Light-touch cleanup only. Do not strip substantive geography.
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Quick examples
for x in ["St. Helens", "Cheshire West & Chester", "Alport & South West Parishes"]:
    print(x, "=> council:", clean_council_name(x), "| ward:", clean_ward_name(x))

St. Helens => council: st helens | ward: st helens
Cheshire West & Chester => council: cheshire west and chester | ward: cheshire west and chester
Alport & South West Parishes => council: alport and south west parishes | ward: alport and south west parishes


## 4. Prepare source rows and candidate ward rows

Each source result area receives cleaned council and ward names. Each candidate ward from the lookup also receives cleaned LAD and ward names.

In [7]:
source = name_2023.copy()

source["clean_council_name"] = source["council_name"].map(clean_council_name)
source["clean_ward_name"] = source["ward_name"].map(clean_ward_name)

ward_candidates["clean_lad_name"] = ward_candidates["LAD23NM"].map(clean_council_name)
ward_candidates["clean_ward_name"] = ward_candidates["WD23NM"].map(clean_ward_name)

# Council candidate list
lad_candidates = (
    ward_candidates[["LAD23CD", "LAD23NM", "clean_lad_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Source result areas:", len(source))
print("Candidate LADs:", len(lad_candidates))
print("Candidate wards:", len(ward_candidates))

display(source[["council_name", "ward_name", "clean_council_name", "clean_ward_name"]].head())
display(ward_candidates.head())

Source result areas: 4831
Candidate LADs: 318
Candidate wards: 7608


,council_name,ward_name,clean_council_name,clean_ward_name
0,Amber Valley,ALFRETON,amber valley,alfreton
1,Amber Valley,Alport & South West Parishes,amber valley,alport and south west parishes
2,Amber Valley,Belper East,amber valley,belper east
3,Amber Valley,Belper North,amber valley,belper north
4,Amber Valley,BELPER SOUTH,amber valley,belper south


,LAD23CD,LAD23NM,WD23CD,WD23NM,clean_lad_name,clean_ward_name
0,E06000001,Hartlepool,E05013038,Burn Valley,hartlepool,burn valley
26,E06000001,Hartlepool,E05013039,De Bruce,hartlepool,de bruce
50,E06000001,Hartlepool,E05013049,Victoria,hartlepool,victoria
73,E06000002,Middlesbrough,E05009853,Acklam,middlesbrough,acklam
90,E06000002,Middlesbrough,E05009854,Ayresome,middlesbrough,ayresome


## 5. Matching engine

The matching engine works in two stages:

1. Match source council name to 2023 LAD name.
2. Match source ward name to 2023 ward name within the suggested LAD.

Auto-approval is deliberately conservative.

In [8]:
def best_fuzzy_match(query, choices, scorer=None):
    """Return best and second-best fuzzy match tuples.

    Output: (best_value, best_score, second_value, second_score)
    """
    query = str(query or "")
    choices = [str(c) for c in choices if str(c) != ""]

    if not choices:
        return None, 0, None, 0

    if RAPIDFUZZ_AVAILABLE:
        if scorer is None:
            scorer = fuzz.WRatio
        matches = process.extract(query, choices, scorer=scorer, limit=2)
        best_value, best_score = matches[0][0], float(matches[0][1])
        if len(matches) > 1:
            second_value, second_score = matches[1][0], float(matches[1][1])
        else:
            second_value, second_score = None, 0
        return best_value, best_score, second_value, second_score

    # Fallback if rapidfuzz is unavailable.
    scored = []
    for c in choices:
        score = difflib.SequenceMatcher(None, query, c).ratio() * 100
        scored.append((c, score))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    best_value, best_score = scored[0]
    if len(scored) > 1:
        second_value, second_score = scored[1]
    else:
        second_value, second_score = None, 0
    return best_value, best_score, second_value, second_score


def match_one_result_area(row):
    src_council = row["clean_council_name"]
    src_ward = row["clean_ward_name"]

    # --------------------------------------------------------
    # Council / LAD match
    # --------------------------------------------------------
    exact_lad = lad_candidates[lad_candidates["clean_lad_name"].eq(src_council)]

    if len(exact_lad) == 1:
        chosen_lad = exact_lad.iloc[0]
        council_match_method = "exact_clean_council"
        council_match_score = 100.0
        council_second_score = 0.0
    elif len(exact_lad) > 1:
        # Extremely rare. Use first but mark ambiguity later.
        chosen_lad = exact_lad.iloc[0]
        council_match_method = "exact_clean_council_ambiguous"
        council_match_score = 100.0
        council_second_score = 100.0
    else:
        best, score, second, second_score = best_fuzzy_match(
            src_council,
            lad_candidates["clean_lad_name"].tolist()
        )
        match_lad = lad_candidates[lad_candidates["clean_lad_name"].eq(best)]
        chosen_lad = match_lad.iloc[0] if len(match_lad) else pd.Series(dtype=object)
        council_match_method = "fuzzy_council"
        council_match_score = score
        council_second_score = second_score

    if chosen_lad.empty:
        return pd.Series({
            "suggested_LAD23CD": np.nan,
            "suggested_LAD23NM": np.nan,
            "suggested_WD23CD": np.nan,
            "suggested_WD23NM": np.nan,
            "council_match_method": "no_council_match",
            "council_match_score": 0,
            "council_second_score": 0,
            "ward_match_method": "no_ward_match",
            "ward_match_score": 0,
            "ward_second_score": 0,
            "ward_match_margin": 0,
            "auto_match_status": "no_council_match",
            "auto_approved": False,
        })

    lad_code = chosen_lad["LAD23CD"]
    lad_name = chosen_lad["LAD23NM"]

    ward_pool = ward_candidates[ward_candidates["LAD23CD"].eq(lad_code)].copy()

    # --------------------------------------------------------
    # Ward match within selected LAD
    # --------------------------------------------------------
    exact_ward = ward_pool[ward_pool["clean_ward_name"].eq(src_ward)]

    if len(exact_ward) == 1:
        chosen_ward = exact_ward.iloc[0]
        ward_match_method = "exact_clean_ward"
        ward_match_score = 100.0
        ward_second_score = 0.0
        ward_match_margin = 100.0
    elif len(exact_ward) > 1:
        chosen_ward = exact_ward.iloc[0]
        ward_match_method = "exact_clean_ward_ambiguous"
        ward_match_score = 100.0
        ward_second_score = 100.0
        ward_match_margin = 0.0
    else:
        best, score, second, second_score = best_fuzzy_match(
            src_ward,
            ward_pool["clean_ward_name"].tolist()
        )
        ward_match_margin = score - second_score
        match_ward = ward_pool[ward_pool["clean_ward_name"].eq(best)]
        chosen_ward = match_ward.iloc[0] if len(match_ward) else pd.Series(dtype=object)
        ward_match_method = "fuzzy_ward"
        ward_match_score = score
        ward_second_score = second_score

    if chosen_ward.empty:
        suggested_wd_code = np.nan
        suggested_wd_name = np.nan
    else:
        suggested_wd_code = chosen_ward["WD23CD"]
        suggested_wd_name = chosen_ward["WD23NM"]

    # --------------------------------------------------------
    # Conservative auto-approval rules
    # --------------------------------------------------------
    if council_match_score >= 99 and ward_match_method == "exact_clean_ward":
        auto_status = "auto_approved_exact"
        auto_approved = True
    elif (
        council_match_score >= 95
        and ward_match_score >= 97
        and ward_match_margin >= 5
        and "ambiguous" not in council_match_method
        and "ambiguous" not in ward_match_method
    ):
        auto_status = "auto_approved_high_confidence_fuzzy"
        auto_approved = True
    elif ward_match_score >= 90 and council_match_score >= 90:
        auto_status = "review_required_possible_match"
        auto_approved = False
    else:
        auto_status = "review_required_low_confidence"
        auto_approved = False

    return pd.Series({
        "suggested_LAD23CD": lad_code,
        "suggested_LAD23NM": lad_name,
        "suggested_WD23CD": suggested_wd_code,
        "suggested_WD23NM": suggested_wd_name,
        "council_match_method": council_match_method,
        "council_match_score": council_match_score,
        "council_second_score": council_second_score,
        "ward_match_method": ward_match_method,
        "ward_match_score": ward_match_score,
        "ward_second_score": ward_second_score,
        "ward_match_margin": ward_match_margin,
        "auto_match_status": auto_status,
        "auto_approved": auto_approved,
    })

## 6. Run matching nationally

This can take a minute or two because it checks thousands of result areas.

In [9]:
match_results = source.apply(match_one_result_area, axis=1)

matches = pd.concat([source.reset_index(drop=True), match_results.reset_index(drop=True)], axis=1)

print("Rows matched:", len(matches))
print("Auto approval breakdown:")
display(matches["auto_match_status"].value_counts(dropna=False).reset_index(name="count"))

print("Council match methods:")
display(matches["council_match_method"].value_counts(dropna=False).reset_index(name="count"))

print("Ward match methods:")
display(matches["ward_match_method"].value_counts(dropna=False).reset_index(name="count"))

display(matches.head())

Rows matched: 4831
Auto approval breakdown:


,auto_match_status,count
0,auto_approved_exact,4756
1,auto_approved_high_confidence_fuzzy,48
2,review_required_possible_match,27


Council match methods:


,council_match_method,count
0,exact_clean_council,4796
1,fuzzy_council,35


Ward match methods:


,ward_match_method,count
0,exact_clean_ward,4791
1,fuzzy_ward,40


,result_area_key,source_year,election_year,election_date,source_file,council_name,lad_code,ward_name,standard_ward_name,ward_code,ec_ward_code,boundary_year,geography_type,atlas_join_strategy,election_type,ordinary_or_by_election,seats_available,electorate,turnout,valid_votes,ballots,invalid_votes,candidate_count,top_party_by_votes,runner_up_party_by_votes,top_party_votes,runner_up_party_votes,margin_votes,margin_pct,top_party_vote_share,runner_up_vote_share,elected_parties,elected_candidates,lowest_elected_votes,first_losing_candidate,first_losing_party,first_losing_votes,con_votes,lab_votes,ld_votes,green_votes,reform_ukip_brexit_votes,independent_votes,sdp_votes,other_votes,con_share,lab_share,ld_share,green_share,reform_ukip_brexit_share,independent_share,sdp_share,other_share,party_fragmentation_index,effective_number_of_parties,data_quality_flag,manual_review_required,candidate_effective_total_votes,source_geography_code,source_geography_name,source_boundary_year,detected_geography_type,atlas_join_ready,join_readiness_status,recommended_next_action,source_code_found_in_lookup,clean_council_name,clean_ward_name,suggested_LAD23CD,suggested_LAD23NM,suggested_WD23CD,suggested_WD23NM,council_match_method,council_match_score,council_second_score,ward_match_method,ward_match_score,ward_second_score,ward_match_margin,auto_match_status,auto_approved
0,2023|NAME|AMBER_VALLEY|ALFRETON,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,ALFRETON,ALFRETON,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,3.0,6673.0,24.7,1675.0,NaN,NaN,13,Labour,Conservative,828,443.0,385.0,0.229851,0.494328,0.264478,Labour; Labour; Labour,"Marshall-Clarke, S.; Dolman, G.; Wood, K.",763,"Boyce, C.",Conservative,443.0,443,828,104,157,143,0,0,0,0.264478,0.494328,0.062090,0.093731,0.085373,0.0,0.0,0.000000,0.665762,2.991878,needs_ward_code_match,True,1675,NaN,ALFRETON,2023,name_only_no_ons_code,False,name_match_required,review_ward_name_dictionary,False,amber valley,alfreton,E07000032,Amber Valley,E05014690,Alfreton,exact_clean_council,100.0,0.0,exact_clean_ward,100.0,0.0,100.0,auto_approved_exact,True
1,2023|NAME|AMBER_VALLEY|ALPORT_AND_SOUTH_WEST_P...,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,Alport & South West Parishes,Alport & South West Parishes,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,2.0,4569.0,41.9,2084.0,NaN,NaN,8,Conservative,Green,868,496.0,372.0,0.178503,0.416507,0.238004,Conservative; Conservative,"Taylor, D.; Orton, J.",846,"Meynell, G.",Green,496.0,868,455,165,496,100,0,0,0,0.416507,0.218330,0.079175,0.238004,0.047985,0.0,0.0,0.000000,0.713637,3.492073,needs_ward_code_match,True,2084,NaN,Alport & South West Parishes,2023,name_only_no_ons_code,False,name_match_required,review_ward_name_dictionary,False,amber valley,alport and south west parishes,E07000032,Amber Valley,E05014691,Alport & South West Parishes,exact_clean_council,100.0,0.0,exact_clean_ward,100.0,0.0,100.0,auto_approved_exact,True
2,2023|NAME|AMBER_VALLEY|BELPER_EAST,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,Belper East,Belper East,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA,ordinary,3.0,8046.0,40.1,3438.0,NaN,NaN,15,Labour,BELPER IND,978,890.0,88.0,0.025596,0.284468,0.258871,Labour; Labour; BELPER IND,"Hill, T.; Porter, J.; Atkinson, F.",890,"Nelson, J.",Conservative,867.0,867,978,234,469,0,0,0,890,0.252182,0.284468,0.068063,0.136417,0.000000,0.0,0.0,0.258871,0.765226,4.259419,needs_ward_code_match,True,3438,NaN,Belper East,2023,name_only_no_ons_code,False,name_match_required,review_ward_name_dictionary,False,amber valley,belper east,E07000032,Amber Valley,E05014692,Belper East,exact_clean_council,100.0,0.0,exact_clean_ward,100.0,0.0,100.0,auto_approved_exact,True
3,2023|NAME|AMBER_VALLEY|BELPER_NORTH,2023,2023,2023-05-04,LEH-Candidates-2023.xlsx,Amber Valley,NaN,Belper North,Belper North,NaN,NaN,2023,name_only_no_ons_code,needs_ward_name_matching,SDA

## 7. Add manual review columns

The review file can be edited manually. For a manual approval, set `approved = TRUE` and either keep the suggested codes or fill the `manual_*` columns.

In [10]:
review_cols = [
    "result_area_key",
    "source_year",
    "election_year",
    "council_name",
    "ward_name",
    "clean_council_name",
    "clean_ward_name",
    "suggested_LAD23CD",
    "suggested_LAD23NM",
    "suggested_WD23CD",
    "suggested_WD23NM",
    "council_match_method",
    "council_match_score",
    "ward_match_method",
    "ward_match_score",
    "ward_second_score",
    "ward_match_margin",
    "auto_match_status",
    "auto_approved",
]

review = matches[review_cols].copy()

review["approved"] = review["auto_approved"]
review["manual_LAD23CD"] = ""
review["manual_LAD23NM"] = ""
review["manual_WD23CD"] = ""
review["manual_WD23NM"] = ""
review["review_notes"] = ""

review["final_LAD23CD"] = np.where(review["approved"], review["suggested_LAD23CD"], "")
review["final_LAD23NM"] = np.where(review["approved"], review["suggested_LAD23NM"], "")
review["final_WD23CD"] = np.where(review["approved"], review["suggested_WD23CD"], "")
review["final_WD23NM"] = np.where(review["approved"], review["suggested_WD23NM"], "")

review_all_path = OUTPUT_DIR / "election_2023_name_match_review_all_v1.csv"
review_auto_path = OUTPUT_DIR / "election_2023_auto_approved_ward_matches_v1.csv"
review_manual_path = OUTPUT_DIR / "election_2023_manual_review_required_v1.csv"

review.to_csv(review_all_path, index=False)
review[review["approved"].eq(True)].to_csv(review_auto_path, index=False)
review[~review["approved"].eq(True)].to_csv(review_manual_path, index=False)

print("Saved:", review_all_path)
print("Saved:", review_auto_path)
print("Saved:", review_manual_path)

print("Auto approved:", review["approved"].sum())
print("Manual review required:", (~review["approved"].eq(True)).sum())

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\name_matching_2023_v1\election_2023_name_match_review_all_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\name_matching_2023_v1\election_2023_auto_approved_ward_matches_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\name_matching_2023_v1\election_2023_manual_review_required_v1.csv
Auto approved: 4804
Manual review required: 27


## 8. Produce top-three fuzzy candidates for manual review rows

This helps manual checking by showing alternatives within the suggested LAD.

In [11]:
def top_ward_candidates_for_row(row, n=3):
    lad_code = row["suggested_LAD23CD"]
    src_ward = row["clean_ward_name"]

    pool = ward_candidates[ward_candidates["LAD23CD"].eq(lad_code)].copy()
    if len(pool) == 0:
        return []

    choices = pool["clean_ward_name"].tolist()

    if RAPIDFUZZ_AVAILABLE:
        extracted = process.extract(src_ward, choices, scorer=fuzz.WRatio, limit=n)
        out = []
        for clean_name, score, _ in extracted:
            rows = pool[pool["clean_ward_name"].eq(clean_name)]
            if len(rows):
                r = rows.iloc[0]
                out.append({
                    "WD23CD": r["WD23CD"],
                    "WD23NM": r["WD23NM"],
                    "score": float(score),
                })
        return out

    scored = []
    for _, r in pool.iterrows():
        score = difflib.SequenceMatcher(None, src_ward, r["clean_ward_name"]).ratio() * 100
        scored.append({"WD23CD": r["WD23CD"], "WD23NM": r["WD23NM"], "score": score})
    return sorted(scored, key=lambda x: x["score"], reverse=True)[:n]

manual = review[~review["approved"].eq(True)].copy()

candidate_rows = []
for _, row in manual.iterrows():
    top = top_ward_candidates_for_row(row, n=3)
    base = row.to_dict()
    for i, cand in enumerate(top, start=1):
        base[f"candidate_{i}_WD23CD"] = cand["WD23CD"]
        base[f"candidate_{i}_WD23NM"] = cand["WD23NM"]
        base[f"candidate_{i}_score"] = cand["score"]
    candidate_rows.append(base)

manual_with_candidates = pd.DataFrame(candidate_rows)

manual_candidates_path = OUTPUT_DIR / "election_2023_manual_review_with_top_candidates_v1.csv"
manual_with_candidates.to_csv(manual_candidates_path, index=False)

print("Saved:", manual_candidates_path)
display(manual_with_candidates.head())

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\name_matching_2023_v1\election_2023_manual_review_with_top_candidates_v1.csv


,result_area_key,source_year,election_year,council_name,ward_name,clean_council_name,clean_ward_name,suggested_LAD23CD,suggested_LAD23NM,suggested_WD23CD,suggested_WD23NM,council_match_method,council_match_score,ward_match_method,ward_match_score,ward_second_score,ward_match_margin,auto_match_status,auto_approved,approved,manual_LAD23CD,manual_LAD23NM,manual_WD23CD,manual_WD23NM,review_notes,final_LAD23CD,final_LAD23NM,final_WD23CD,final_WD23NM,candidate_1_WD23CD,candidate_1_WD23NM,candidate_1_score,candidate_2_WD23CD,candidate_2_WD23NM,candidate_2_score,candidate_3_WD23CD,candidate_3_WD23NM,candidate_3_score
0,2023|NAME|BASILDON|ST_MARTINS,2023,2023,Basildon,St. Martins,basildon,st martins,E07000066,Basildon,E05004032,St Martin's,exact_clean_council,100.0,fuzzy_ward,95.238095,45.0,50.238095,review_required_possible_match,False,False,,,,,,,,,,E05004032,St Martin's,95.238095,E05004030,Pitsea North West,45.0,E05004031,Pitsea South East,45.000000
1,2023|NAME|CAMBRIDGE|QUEEN_EDITHS,2023,2023,Cambridge,Queen Ediths,cambridge,queen ediths,E07000008,Cambridge,E05013060,Queen Edith's,exact_clean_council,100.0,fuzzy_ward,96.000000,40.0,56.000000,review_required_possible_match,False,False,,,,,,,,,,E05013060,Queen Edith's,96.000000,E05013056,King's Hedges,40.0,E05013058,Newnham,38.571429
2,2023|NAME|CASTLE_POINT|ST_GEORGES,2023,2023,Castle Point,St. Georges,castle point,st georges,E07000069,Castle Point,E05004091,St George's,exact_clean_council,100.0,fuzzy_ward,95.238095,60.0,35.238095,review_required_possible_match,False,False,,,,,,,,,,E05004091,St George's,95.238095,E05004094,St Peter's,60.0,E05004092,St James,55.555556
3,2023|NAME|CASTLE_POINT|ST_MARYS,2023,2023,Castle Point,St. Marys,castle point,st marys,E07000069,Castle Point,E05004093,St Mary's,exact_clean_council,100.0,fuzzy_ward,94.117647,62.5,31.617647,review_required_possible_match,False,False,,,,,,,,,,E05004093,St Mary's,94.117647,E05004092,St James,62.5,E05004094,St Peter's,55.555556
4,2023|NAME|CASTLE_POINT|ST_PETERS,2023,2023,Castle Point,St. Peters,castle point,st peters,E07000069,Castle Point,E05004094,St Peter's,exact_clean_council,100.0,fuzzy_ward,94.736842,60.0,34.736842,review_required_possible_match,False,False,,,,,,,,,,E05004094,St Peter's,94.736842,E05004091,St George's,60.0,E05004092,St James,58.823529


## 9. Create ward-name dictionary suggestions

This does not overwrite your existing dictionary. It writes a suggested V2 file which can be reviewed.

In [12]:
ward_dict_cols = [
    "source_year",
    "council_name",
    "ward_name",
    "clean_council_name",
    "clean_ward_name",
    "suggested_LAD23CD",
    "suggested_LAD23NM",
    "suggested_WD23CD",
    "suggested_WD23NM",
    "match_method",
    "match_score",
    "approved",
    "manual_LAD23CD",
    "manual_LAD23NM",
    "manual_WD23CD",
    "manual_WD23NM",
    "notes",
]

suggestions = pd.DataFrame({
    "source_year": review["source_year"],
    "council_name": review["council_name"],
    "ward_name": review["ward_name"],
    "clean_council_name": review["clean_council_name"],
    "clean_ward_name": review["clean_ward_name"],
    "suggested_LAD23CD": review["suggested_LAD23CD"],
    "suggested_LAD23NM": review["suggested_LAD23NM"],
    "suggested_WD23CD": review["suggested_WD23CD"],
    "suggested_WD23NM": review["suggested_WD23NM"],
    "match_method": review["auto_match_status"],
    "match_score": review["ward_match_score"],
    "approved": review["approved"],
    "manual_LAD23CD": "",
    "manual_LAD23NM": "",
    "manual_WD23CD": "",
    "manual_WD23NM": "",
    "notes": "",
})

suggestions = suggestions.drop_duplicates(subset=["source_year", "council_name", "ward_name"])

suggestions_path = DICTIONARY_DIR / "ward_name_dictionary_v2_suggestions.csv"
suggestions.to_csv(suggestions_path, index=False)

print("Saved:", suggestions_path)
display(suggestions.head())

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\dictionaries\ward_name_dictionary_v2_suggestions.csv


,source_year,council_name,ward_name,clean_council_name,clean_ward_name,suggested_LAD23CD,suggested_LAD23NM,suggested_WD23CD,suggested_WD23NM,match_method,match_score,approved,manual_LAD23CD,manual_LAD23NM,manual_WD23CD,manual_WD23NM,notes
0,2023,Amber Valley,ALFRETON,amber valley,alfreton,E07000032,Amber Valley,E05014690,Alfreton,auto_approved_exact,100.0,True,,,,,
1,2023,Amber Valley,Alport & South West Parishes,amber valley,alport and south west parishes,E07000032,Amber Valley,E05014691,Alport & South West Parishes,auto_approved_exact,100.0,True,,,,,
2,2023,Amber Valley,Belper East,amber valley,belper east,E07000032,Amber Valley,E05014692,Belper East,auto_approved_exact,100.0,True,,,,,
3,2023,Amber Valley,Belper North,amber valley,belper north,E07000032,Amber Valley,E05014693,Belper North,auto_approved_exact,100.0,True,,,,,
4,2023,Amber Valley,BELPER SOUTH,amber valley,belper south,E07000032,Amber Valley,E05014694,Belper South,auto_approved_exact,100.0,True,,,,,


## 10. Apply approved matches to the ward-result summary

This cell reads the review file from disk, so it can be rerun after manual edits.

Rules:

- If `manual_*` columns are filled, they override the suggested values.
- If not, suggested values are used for rows marked `approved == TRUE`.
- The original `ward_result_summary_v1.csv` is not overwritten.
- Output is `ward_result_summary_v2_with_2023_matches.csv`.

In [15]:
REVIEW_TO_APPLY_PATH = OUTPUT_DIR / "election_2023_name_match_review_all_v1.csv"

review_apply = pd.read_csv(REVIEW_TO_APPLY_PATH, low_memory=False)

# Robust boolean parsing for approved column.
review_apply["approved"] = (
    review_apply["approved"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes", "y"])
)

approved = review_apply[review_apply["approved"]].copy()

print("Approved rows to apply:", len(approved))

# Manual values override suggestions if present.
def choose_final(row, manual_col, suggested_col):
    manual = row.get(manual_col, "")
    if pd.notna(manual) and str(manual).strip() != "":
        return str(manual).strip()
    return row.get(suggested_col, np.nan)

approved["final_LAD23CD"] = approved.apply(lambda r: choose_final(r, "manual_LAD23CD", "suggested_LAD23CD"), axis=1)
approved["final_LAD23NM"] = approved.apply(lambda r: choose_final(r, "manual_LAD23NM", "suggested_LAD23NM"), axis=1)
approved["final_WD23CD"] = approved.apply(lambda r: choose_final(r, "manual_WD23CD", "suggested_WD23CD"), axis=1)
approved["final_WD23NM"] = approved.apply(lambda r: choose_final(r, "manual_WD23NM", "suggested_WD23NM"), axis=1)

# Basic validation: approved rows must have final codes.
missing_final = approved[
    approved[["final_LAD23CD", "final_LAD23NM", "final_WD23CD", "final_WD23NM"]]
    .isna()
    .any(axis=1)
]

if len(missing_final) > 0:
    display(missing_final.head(20))
    raise ValueError("Some approved rows do not have final LAD/ward codes or names.")

ward_summary = pd.read_csv(WARD_SUMMARY_PATH, low_memory=False)

patched = ward_summary.copy()

patch_cols = approved[[
    "result_area_key",
    "final_LAD23CD",
    "final_LAD23NM",
    "final_WD23CD",
    "final_WD23NM",
    "auto_match_status",
    "ward_match_score",
    "council_match_score",
]].copy()

patched = patched.merge(
    patch_cols,
    on="result_area_key",
    how="left",
    validate="one_to_one"
)

match_mask = patched["final_WD23CD"].notna()

print("Rows patched:", match_mask.sum())

# Patch geography fields for approved 2023 matches.
patched.loc[match_mask, "lad_code"] = patched.loc[match_mask, "final_LAD23CD"]
patched.loc[match_mask, "ward_code"] = patched.loc[match_mask, "final_WD23CD"]
patched.loc[match_mask, "source_geography_code"] = patched.loc[match_mask, "final_WD23CD"]
patched.loc[match_mask, "source_geography_name"] = patched.loc[match_mask, "final_WD23NM"]
patched.loc[match_mask, "source_boundary_year"] = 2023
patched.loc[match_mask, "detected_geography_type"] = "electoral_ward_or_division"
patched.loc[match_mask, "atlas_join_ready"] = True
patched.loc[match_mask, "join_readiness_status"] = "ward_oa21_lookup_available_name_matched"
patched.loc[match_mask, "recommended_next_action"] = "ready_for_oa21_crosswalk"
patched.loc[match_mask, "source_code_found_in_lookup"] = True
patched.loc[match_mask, "manual_review_required"] = False
patched.loc[match_mask, "atlas_join_strategy"] = "oa21_to_wd23_name_matched"
patched.loc[match_mask, "data_quality_flag"] = (
    patched.loc[match_mask, "data_quality_flag"].fillna("").astype(str)
    + ";2023_ward_name_matched"
)
patched.loc[match_mask, "name_match_status"] = patched.loc[match_mask, "auto_match_status"]
patched.loc[match_mask, "name_match_score"] = patched.loc[match_mask, "ward_match_score"]

# Drop helper columns from final output.
helper_cols = [
    "final_LAD23CD", "final_LAD23NM", "final_WD23CD", "final_WD23NM",
    "auto_match_status", "ward_match_score", "council_match_score",
]
patched = patched.drop(columns=[c for c in helper_cols if c in patched.columns])

out_path = ELECTION_DIR / "ward_result_summary_v2_with_2023_matches.csv"
patched.to_csv(out_path, index=False)

print("Saved:", out_path)

print("New join-readiness breakdown for 2023:")
display(
    patched[patched["source_year"].astype(str).eq("2023")]
    .groupby(["join_readiness_status", "detected_geography_type"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

Approved rows to apply: 4831
Rows patched: 4831
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\ward_result_summary_v2_with_2023_matches.csv
New join-readiness breakdown for 2023:


,join_readiness_status,detected_geography_type,rows
0,ward_oa21_lookup_available_name_matched,electoral_ward_or_division,4831


## 11. What to do after this notebook

After reviewing and applying matches:

1. Use `ward_result_summary_v2_with_2023_matches.csv` as the input to Notebook 13.
2. If Notebook 13 has a fixed path to `ward_result_summary_v1.csv`, change it to `ward_result_summary_v2_with_2023_matches.csv`.
3. Rerun Notebooks 13, 14 and 15.
4. Inspect whether North West and national election coverage improved.

Do not delete `ward_result_summary_v1.csv`. Keep it as the original pre-match baseline.